# 02b - Unstandardized Chronological Dataset

Rebuild the modeling data before Week 5 model comparison. This version preserves raw numeric values and raw categorical values: it does **not** standardize numeric columns and does **not** frequency-encode categories.

Mandatory split:
- **Train:** February 2025 through January 2026
- **Validation:** February 2026 through April 2026
- **Test:** May 2026 only

The May 2026 rows are exported but must remain untouched until a model and feature set have been selected using validation performance.

## Design decisions carried forward from earlier notebooks

- Keep only `Residential` / `SingleFamilyResidence` transactions.
- Keep `10,000 <= ClosePrice <= 20,000,000`.
- Require a valid close date and latitude/longitude.
- Convert boolean fields to `0/1` and blank categorical values to `Unknown`.
- Add `HomeAge = 2026 - YearBuilt`.
- Preserve missing numeric values. Train-only imputation belongs in the later modeling pipeline.
- Preserve raw categories so one-hot encoding or another encoding can be compared later.
- Retain price-related candidate columns, but treat `ListPrice` and `DaysOnMarket` as potentially unavailable/leaky depending on when a prediction is supposed to be made.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_DIR = Path("data")
FULL_OUTPUT_FILE = DATA_DIR / "week5_unstandardized_model_data.csv"
VALIDATION_OUTPUT_FILE = DATA_DIR / "week5_validation_unstandardized.csv"

CURRENT_YEAR = 2026
MIN_CLOSE_PRICE = 10_000
MAX_CLOSE_PRICE = 20_000_000

TRAIN_MONTHS = pd.period_range("2025-02", "2026-01", freq="M").astype(str).tolist()
VALIDATION_MONTHS = pd.period_range("2026-02", "2026-04", freq="M").astype(str).tolist()
TEST_MONTHS = ["2026-05"]
REQUIRED_MONTHS = TRAIN_MONTHS + VALIDATION_MONTHS + TEST_MONTHS

id_date_cols = ["ListingId", "CloseDate"]
filter_cols = ["PropertyType", "PropertySubType"]
numeric_features = [
    "Latitude", "Longitude", "LivingArea", "BuildingAreaTotal",
    "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt", "Stories",
    "GarageSpaces", "ParkingTotal", "AssociationFee",
    "LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea",
    "MainLevelBedrooms", "OriginalListPrice", "ListPrice", "DaysOnMarket",
]
boolean_features = [
    "ViewYN", "WaterfrontYN", "PoolPrivateYN", "AttachedGarageYN",
    "FireplaceYN", "NewConstructionYN",
]
categorical_features = [
    "City", "PostalCode", "CountyOrParish", "MLSAreaMajor",
    "HighSchoolDistrict", "Levels",
]
target = "ClosePrice"
selected_cols = (
    id_date_cols + filter_cols + numeric_features + boolean_features
    + categorical_features + [target]
)

## Load exactly the required monthly files

In [3]:
monthly_files = {
    month: DATA_DIR / f"CRMLSSold{month.replace('-', '')}.csv"
    for month in REQUIRED_MONTHS
}
missing_files = [str(path) for path in monthly_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required monthly files: {missing_files}")

frames = []
for source_month, path in monthly_files.items():
    available_cols = pd.read_csv(path, nrows=0).columns
    missing_cols = sorted(set(selected_cols) - set(available_cols))
    if missing_cols:
        raise ValueError(f"{path.name} is missing required columns: {missing_cols}")

    frame = pd.read_csv(
        path, usecols=selected_cols, dtype=str, keep_default_na=False, low_memory=False
    )
    frame["source_month"] = source_month
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(monthly_files)} files and {len(raw):,} raw rows.")
print(f"Required coverage: {REQUIRED_MONTHS[0]} through {REQUIRED_MONTHS[-1]}")

Loaded 16 files and 345,232 raw rows.
Required coverage: 2025-02 through 2026-05


## Clean values without scaling or category encoding

In [4]:
work = raw[
    (raw["PropertyType"] == "Residential")
    & (raw["PropertySubType"] == "SingleFamilyResidence")
].copy()
rows_after_property_filter = len(work)

work["CloseDate"] = pd.to_datetime(work["CloseDate"], errors="coerce")
work["close_month"] = work["CloseDate"].dt.to_period("M").astype(str)

for col in numeric_features + [target]:
    work[col] = pd.to_numeric(work[col].replace("", np.nan), errors="coerce")

work = work[work[target].between(MIN_CLOSE_PRICE, MAX_CLOSE_PRICE)].copy()
rows_after_target_filter = len(work)
work = work.dropna(subset=["CloseDate", "Latitude", "Longitude"]).copy()
rows_after_required_filter = len(work)

for col in boolean_features:
    normalized = work[col].astype(str).str.strip().str.lower()
    work[col] = (
        normalized
        .map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0})
        .fillna(0)
        .astype(int)
    )

for col in categorical_features:
    work[col] = work[col].astype(str).str.strip().replace("", "Unknown")
work["PostalCode"] = (
    work["PostalCode"].str.extract(r"(\d{5})", expand=False).fillna("Unknown")
)

work["HomeAge"] = CURRENT_YEAR - work["YearBuilt"]
work.loc[(work["HomeAge"] < 0) | (work["HomeAge"] > 250), "HomeAge"] = np.nan

duplicate_rows = int(work.duplicated().sum())
work = work.drop_duplicates().copy()

print(f"After property filter: {rows_after_property_filter:,}")
print(f"After ClosePrice filter: {rows_after_target_filter:,}")
print(f"After date/coordinate filter: {rows_after_required_filter:,}")
print(f"Exact duplicate rows removed: {duplicate_rows:,}")

After property filter: 173,338
After ClosePrice filter: 173,202
After date/coordinate filter: 173,186
Exact duplicate rows removed: 8


## Apply and verify the mandatory split

In [5]:
month_to_split = (
    {month: "train" for month in TRAIN_MONTHS}
    | {month: "validation" for month in VALIDATION_MONTHS}
    | {month: "test" for month in TEST_MONTHS}
)
work["split"] = work["close_month"].map(month_to_split)
model_data = work[work["split"].notna()].copy()

output_cols = (
    ["ListingId", "CloseDate", "close_month", "split"]
    + numeric_features + ["HomeAge"] + boolean_features + categorical_features + [target]
)
model_data = model_data[output_cols].sort_values(["CloseDate", "ListingId"]).reset_index(drop=True)

expected_by_split = {
    "train": TRAIN_MONTHS,
    "validation": VALIDATION_MONTHS,
    "test": TEST_MONTHS,
}
for split_name, expected_months in expected_by_split.items():
    actual_months = sorted(model_data.loc[model_data["split"] == split_name, "close_month"].unique())
    assert actual_months == expected_months, (split_name, actual_months, expected_months)

assert not any(col.endswith("_scaled") for col in model_data.columns)
assert not any(col.endswith("_freq") for col in model_data.columns)
assert model_data["split"].isna().sum() == 0
assert model_data[target].between(MIN_CLOSE_PRICE, MAX_CLOSE_PRICE).all()
assert model_data.duplicated().sum() == 0

split_summary = (
    model_data.groupby("split", sort=False)
    .agg(
        rows=("ListingId", "size"),
        first_month=("close_month", "min"),
        last_month=("close_month", "max"),
        month_count=("close_month", "nunique"),
    )
)
split_summary

,rows,first_month,last_month,month_count
split,,,,
train,129443,2025-02,2026-01,12
validation,31722,2026-02,2026-04,3
test,12013,2026-05,2026-05,1


## Dataset audit

Missing numeric values are expected here. They are intentionally left for a train-fitted imputer in the modeling notebook.

In [6]:
missing_audit = pd.DataFrame({
    "missing_rows": model_data.isna().sum(),
    "missing_pct": model_data.isna().mean().mul(100),
}).sort_values("missing_pct", ascending=False)
missing_audit[missing_audit["missing_rows"] > 0].round(2)

,missing_rows,missing_pct
BuildingAreaTotal,161686,93.36
MainLevelBedrooms,67893,39.20
AssociationFee,50406,29.11
Stories,18371,10.61
GarageSpaces,6766,3.91
LotSizeSquareFeet,2999,1.73
LotSizeAcres,2999,1.73
LotSizeArea,2984,1.72
OriginalListPrice,359,0.21
HomeAge,108,0.06


In [7]:
validation_data = model_data[model_data["split"] == "validation"].copy()
model_data.to_csv(FULL_OUTPUT_FILE, index=False)
validation_data.to_csv(VALIDATION_OUTPUT_FILE, index=False)

print(f"Wrote {FULL_OUTPUT_FILE} with shape {model_data.shape}")
print(f"Wrote {VALIDATION_OUTPUT_FILE} with shape {validation_data.shape}")
print("No standardized or frequency-encoded columns were exported.")

Wrote data/week5_unstandardized_model_data.csv with shape (173178, 36)
Wrote data/week5_validation_unstandardized.csv with shape (31722, 36)
No standardized or frequency-encoded columns were exported.


## Handoff to modeling

The next notebook should fit preprocessing on the training rows only. A strong leakage-safe first comparison is raw numeric features plus one-hot-encoded location categories. `ListPrice` and `DaysOnMarket` should only be included if the prediction use case confirms they are available at prediction time.